# Sync AP/LFP clock with NI/TTL event clock

## Read the data

Read LFPs and NIs

In [1]:
MAX_HOUR_IN_SECONDS = 60*60*2
# MAX_HOUR_IN_SECONDS = 60*30

In [2]:
from Global_functions import read_file_by_time_steps
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                print(f"File found: {file_path}")
    return matching_files

# Other parameters about what data to read
# chanList = [-1] # list of channels to extract, by index in saved file 
tstep = 60
t_vec = np.arange(0, MAX_HOUR_IN_SECONDS, tstep)

spikeglx_folder = '//132.66.34.210/Pixel1/1_auditory_neuropixels_BarakH/20240319_C9_T2_NP2_-10dB_g0'
binFullPath_LFP = Path(find_files(spikeglx_folder, r'.*\.lf\.bin$')[0])
binFullPath_NI = Path(find_files(spikeglx_folder, r'.*\.nidq\.bin$')[0])

TTL_NI, _ = read_file_by_time_steps(binFullPath_NI, t_vec, tstep, [4])
TTL_NI = np.concatenate(TTL_NI, axis=1)
sync_NI, sr_NI = read_file_by_time_steps(binFullPath_NI, t_vec, tstep, [7])
sync_NI = np.concatenate(sync_NI, axis=1)
sync_LFP, sr_LFP = read_file_by_time_steps(binFullPath_LFP, t_vec, tstep, [-1])
sync_LFP = np.concatenate(sync_LFP, axis=1)

In [3]:
%matplotlib inline

# Plot the first of the extracted channels
tNI = np.linspace(t_vec[0], t_vec[-1] + tstep, TTL_NI.shape[1])
tLFP = np.linspace(t_vec[0], t_vec[-1] + tstep, sync_LFP.shape[1])
# tNI = np.arange(t_vec[0], TTL_NI.shape[1] / sr_NI, 1/sr_NI)
# tLFP = np.arange(t_vec[0], sync_LFP.shape[1] / sr_LFP, 1/sr_LFP)

fig, ax = plt.subplots(3, 1)
ax[0].plot(tNI[0:int(60*sr_NI)], sync_NI[0, 0:int(60*sr_NI)])
ax[0].set_ylabel('Sync NI')
ax[1].plot(tNI[0:int(60*sr_NI)], TTL_NI[0, 0:int(60*sr_NI)])
ax[1].set_ylabel('TTL NI')
ax[2].plot(tLFP[0:int(60*sr_LFP)], sync_LFP[0,0:int(60*sr_LFP)])
ax[2].set_ylabel('Sync LFP')
plt.show(block=False)

In [4]:
from Global_functions import read_sync_times_from_stream
import tqdm


class TtlStampConsts:
    N_BITS = 14
    ZERO_TIME_MS = 1
    BIT_SIZE_MS = 5
    ONSET_TTL_LENGTH_MS = 10

def read_ttl_from_stream(data, t_vec):
    sr = 1 / np.mean(np.diff(t_vec))
    # sr = 1 / (t_vec[1] - t_vec[0])

    nBits = TtlStampConsts.N_BITS
    rampSizeMs = 0
    zeroTimeMs = TtlStampConsts.ZERO_TIME_MS
    bitSizeMs = TtlStampConsts.BIT_SIZE_MS
    onsetTtlSizeMs = TtlStampConsts.ONSET_TTL_LENGTH_MS
    entireStampTimeLengthMs = TtlStampConsts.ONSET_TTL_LENGTH_MS + TtlStampConsts.BIT_SIZE_MS * TtlStampConsts.N_BITS

    nSamplesZero = max(round(sr * zeroTimeMs / 1000), 1)
    nSamplesRamp = round(sr * rampSizeMs / 1000)
    nPeriBitSamples = nSamplesZero + nSamplesRamp
    proportionOfBitIgnoredAtEachEdge = 0.2
    nPeriBitSamples = round(nPeriBitSamples + bitSizeMs / 1000 * sr * proportionOfBitIgnoredAtEachEdge)
    nPeriBitSamples = min(nPeriBitSamples, round(bitSizeMs / 1000 * sr / 2) - 1)

    ttlThreshold = 0.5 * (max(data) + min(data))

    isAboveThreshold = data > ttlThreshold

    # plt.figure(figsize=(20, 10))
    # # plt.plot(t_vec, data)
    # # plt.axhline(ttlThreshold, color='k', linestyle='--')
    # plt.plot(t_vec, isAboveThreshold)
    # plt.show(block=False)


    aboveThresholdIndices = np.where(isAboveThreshold)[0]

    interOnesInterval = np.insert(np.diff(aboveThresholdIndices), 0, len(data) + 1)

    minZeroSamplesBetweenStamps = entireStampTimeLengthMs / 1000 * sr
    isFirstStampBitOne = interOnesInterval > minZeroSamplesBetweenStamps
    firstStampBitIndices = aboveThresholdIndices[isFirstStampBitOne]

    nStamps = len(firstStampBitIndices)
    stampValues = np.full(nStamps, np.nan)
    stampIndices = np.full(nStamps, np.nan)

    for stampInd in range(nStamps):
        currentStampFirstInd = firstStampBitIndices[stampInd]
        bitsForCurrentStamp = np.full(nBits, np.nan)

        for bitInd in range(nBits):
            currentBitFirstIndex = currentStampFirstInd + int(
                np.ceil((onsetTtlSizeMs + (bitInd) * bitSizeMs) / 1000 * sr)) + nPeriBitSamples
            currentBitLastIndex = currentStampFirstInd + int(
                np.floor((onsetTtlSizeMs + (bitInd + 1) * bitSizeMs) / 1000 * sr)) - nPeriBitSamples

            currentBitFractionAboveThreshold = np.mean(isAboveThreshold[currentBitFirstIndex:currentBitLastIndex])

            bitsForCurrentStamp[bitInd] = currentBitFractionAboveThreshold > 0.5

        stampIndices[stampInd] = currentStampFirstInd
        bitsForCurrentStamp = bitsForCurrentStamp[::-1]
        stampValues[stampInd] = int("".join(map(str, bitsForCurrentStamp.astype(int))), 2)

    stampTimesSec = stampIndices / sr

    return stampValues, stampIndices, stampTimesSec


sync_NI_times = read_sync_times_from_stream(np.squeeze(sync_NI), tNI)
sync_LFP_times = read_sync_times_from_stream(np.squeeze(sync_LFP), tLFP)

TTL_NI_stamps, TTL_NI_inds, TTL_NI_times = read_ttl_from_stream(np.squeeze(TTL_NI), tNI)


In [5]:
print(np.unique(TTL_NI_stamps))

In [6]:
del sync_NI, TTL_NI, sync_LFP, tLFP, tNI

In [7]:
save_path = Path(spikeglx_folder + '/Tprime_times')
save_path.mkdir(exist_ok=True)

np.savetxt(save_path / 'NI_sync_times.txt', sync_NI_times)
np.savetxt(save_path / 'LFP_sync_times.txt', sync_LFP_times)
np.savetxt(save_path / 'TTL_times.txt', TTL_NI_times)
np.savetxt(save_path / 'TTL_labels.txt', TTL_NI_stamps)

Read APs

In [8]:
tstep = 30
t_vec = np.arange(0, MAX_HOUR_IN_SECONDS, tstep)

binFullPath_AP = Path(find_files(spikeglx_folder, r'.*\.ap\.bin$')[0])
sync_AP, sr_AP = read_file_by_time_steps(binFullPath_AP, t_vec, tstep, [-1])
sync_AP = np.concatenate(sync_AP, axis=1)

tAP = np.linspace(t_vec[0], t_vec[-1] + tstep, sync_AP.shape[1])

In [9]:
# np.save(spikeglx_folder + 'sync_AP.npy', sync_AP)
# np.save(spikeglx_folder + 'tAP.npy', tAP)

In [10]:
data_stream = sync_AP[0,int(380*sr_AP):int(381*sr_AP)]
t_vec = tAP[int(380*sr_AP):int(381*sr_AP)]
trigger_val = 0.5 * (max(sync_AP[0,0:int(5*sr_AP)]) + min(sync_AP[0,0:int(5*sr_AP)]))
edges_times_mask = np.argwhere((data_stream[:-1] < trigger_val) & (data_stream[1:] > trigger_val))
times = t_vec[edges_times_mask.flatten()]

fig1, ax = plt.subplots(1, 1)
ax.plot(t_vec, data_stream)
ax.axhline(trigger_val)
ax.set_ylabel('Sync AP')
plt.show(block=False)

In [11]:
from Global_functions import read_sync_times_from_stream

sync_AP_times = []
for ii in range(0, sync_AP.shape[1], int(1.5 * sr_AP)):
    sync_AP_times.append(read_sync_times_from_stream(np.squeeze(sync_AP[:,ii:int(ii + 1.5 * sr_AP)]), tAP[ii:int(ii + 1.5 * sr_AP)]))


In [12]:
sync_AP_times = np.concatenate(sync_AP_times)

In [13]:
del sync_AP, tAP

## Save All times and events to text files

In [14]:
save_path = Path(spikeglx_folder + '/Tprime_times')
save_path.mkdir(exist_ok=True)

np.savetxt(save_path / 'NI_sync_times.txt', sync_NI_times)
np.savetxt(save_path / 'LFP_sync_times.txt', sync_LFP_times)
np.savetxt(save_path / 'TTL_times.txt', TTL_NI_times)
np.savetxt(save_path / 'TTL_labels.txt', TTL_NI_stamps)
np.savetxt(save_path / 'AP_sync_times.txt', sync_AP_times)
